# Grad-CAM Demo Using SimpleCNN (CIFAR-10)
This notebook implements Grad-CAM for the SimpleCNN architecture used in class.


## 1. Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

## 2. Define SimpleCNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        self.features = x              # store features for Grad-CAM
        x = x.view(-1, 64 * 8 * 8)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## 3. Load CIFAR-10

In [ ]:
test_transforms = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])

test_set = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_transforms)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=1, shuffle=True)

classes = test_set.classes

## 4. Initialize Model and Load Weights (Optional)

In [ ]:
model = SimpleCNN()
model.eval()

# Uncomment if you trained SimpleCNN earlier:
# model.load_state_dict(torch.load('simplecnn_cifar10.pth', map_location='cpu'))

## 5. Register Backward Hook on Last Conv Layer

In [ ]:
gradients = None

def save_gradients(module, grad_in, grad_out):
    global gradients
    gradients = grad_out[0]

model.conv2.register_backward_hook(save_gradients)

## 6. Grad-CAM Computation

In [ ]:
def compute_gradcam(image_tensor, class_idx=None):
    global gradients

    output = model(image_tensor)

    if class_idx is None:
        class_idx = output.argmax().item()

    model.zero_grad()
    target = output[0, class_idx]
    target.backward()

    feature_maps = model.features.detach()[0]  # shape: [64, H, W]
    grads = gradients.detach()[0]               # shape: [64, H, W]

    weights = grads.mean(dim=[1,2])

    # Build 2D CAM on the same device as feature maps
    cam = torch.zeros(feature_maps.shape[1:], dtype=torch.float32, device=feature_maps.device)
    for i, w in enumerate(weights):
        cam += w * feature_maps[i, :, :]

    cam = torch.relu(cam)
    cam -= cam.min()
    cam /= cam.max()

    return cam.detach().cpu().numpy(), class_idx


## 7. Visualization

In [ ]:
def show_gradcam(image_tensor, cam, class_name):
    img = image_tensor.squeeze().detach().cpu().numpy().transpose(1,2,0)
    img = img * np.array([0.2470,0.2435,0.2616]) + np.array([0.4914,0.4822,0.4465])
    img = np.clip(img, 0, 1)

    # Resize CAM to image size using bilinear interpolation (avoids np.interp 1D limitation)
    cam_tensor = torch.tensor(cam, dtype=torch.float32)[None, None, ...]
    cam_resized = torch.nn.functional.interpolate(cam_tensor, size=img.shape[:2], mode='bilinear', align_corners=False)
    cam_resized = cam_resized.squeeze().cpu().numpy()

    plt.figure(figsize=(6,3))

    plt.subplot(1,2,1)
    plt.title('Original')
    plt.imshow(img)
    plt.axis('off')

    plt.subplot(1,2,2)
    plt.title(f'Grad-CAM ({class_name})')
    plt.imshow(img, alpha=0.6)
    plt.imshow(cam_resized, cmap='jet', alpha=0.4)
    plt.axis('off')

    plt.show()


## 8. Run on a Sample CIFAR-10 Image

In [ ]:
image, label = next(iter(test_loader))
print("True label:", classes[label[0]])

cam, pred_class = compute_gradcam(image)
print("Predicted label:", classes[pred_class])

show_gradcam(image, cam, classes[pred_class])